In [ ]:
!pip install rouge-score bert-score


In [ ]:
import pandas as pd, numpy as np
from rouge_score import rouge_scorer
from bert_score import score as bertscore

# Load reference answers
ref_df = pd.read_csv("dataset_clean_fixed.csv", sep=";", encoding="utf-8-sig")

# Load model outputs
gpt_df = pd.read_csv("dataset_output.csv")
ft_df = pd.read_csv("ft_results.csv")

# Match by id
gpt_eval = gpt_df.merge(ref_df[["id", "correct_answer"]], on="id")
gpt_eval["gpt_response"] = gpt_eval["gpt_response"].fillna("")

ft_eval = ft_df.merge(ref_df[["id", "correct_answer"]], on="id")
ft_eval["answer"] = ft_eval["answer"].fillna("")


In [ ]:
def evaluate(refs, preds, name):
    # Create clean lists for references and predictions
    rs, ps = [], []
    for r, p in zip(refs, preds):
        if p.strip(): #we ignore empty model outputs
            rs.append(r)
            ps.append(p)

    # ROUGE
    # rouge1 = single word overlap
    # rougeL = longest matching word sequence
    sc = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)
    # Store scores for all examples
    r1, rl = [], []
    # Calculate ROUGE score for each prediction-reference pair
    for r, p in zip(rs, ps):
        s = sc.score(r, p)
        r1.append(s['rouge1'].fmeasure)
        rl.append(s['rougeL'].fmeasure)

    # BERTScore
    # returns precision, recall, F1, we use only F1
    _, _, F1 = bertscore(ps, rs, lang='de', verbose=False)

    print("\n", name, ":")
    print("Answered:", len(rs), "/", len(refs))
    print("ROUGE-1:", np.mean(r1))
    print("ROUGE-L:", np.mean(rl))
    print("BERTScore:", round(F1.mean().item(), 4))
#Evaluate GPT model
evaluate(gpt_eval['correct_answer'].tolist(), gpt_eval['gpt_response'].tolist(), 'GPT-4o-mini')
#Evaluate fine-tuned model
evaluate(ft_eval['correct_answer'].tolist(), ft_eval['answer'].tolist(), 'Falcon-RW-1B-FT')